In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install wandb ptflops

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from ptflops import get_model_complexity_info
import wandb
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)


In [3]:
wandb.init(
    project="cifar10-cnn-mlops",
    name="cnn-gradient-flow"
)


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mamta000542 (mamta000542-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
class CIFAR10Custom(Dataset):
    def __init__(self, train=True):
        self.data = torchvision.datasets.CIFAR10(
            root="./data",
            train=train,
            download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
            ])
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return x, y


In [5]:
train_data = CIFAR10Custom(train=True)
test_data  = CIFAR10Custom(train=False)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=64)


100%|██████████| 170M/170M [00:05<00:00, 29.0MB/s]


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64*8*8,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [7]:
model = SimpleCNN()
flops, params = get_model_complexity_info(model, (3,32,32), as_strings=True)
print("FLOPs:", flops)
print("Params:", params)

wandb.log({"FLOPs": flops, "Params": params})


SimpleCNN(
  1.07 M, 100.000% Params, 6.8 MMac, 98.572% MACs, 
  (conv): Sequential(
    19.39 k, 1.811% Params, 5.75 MMac, 83.333% MACs, 
    (0): Conv2d(896, 0.084% Params, 917.5 KMac, 13.295% MACs, 3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(0, 0.000% Params, 32.77 KMac, 0.475% MACs, )
    (2): MaxPool2d(0, 0.000% Params, 32.77 KMac, 0.475% MACs, kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(18.5 k, 1.727% Params, 4.73 MMac, 68.613% MACs, 32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(0, 0.000% Params, 16.38 KMac, 0.237% MACs, )
    (5): MaxPool2d(0, 0.000% Params, 16.38 KMac, 0.237% MACs, kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    1.05 M, 98.189% Params, 1.05 MMac, 15.239% MACs, 
    (0): Linear(1.05 M, 97.949% Params, 1.05 MMac, 15.198% MACs, in_features=4096, out_features=256, bias=True)
    (1): ReLU(0, 0.000% Params, 256.0 Mac, 0.004% MACs, 

In [8]:
def plot_grad_flow(model):
    ave_grads = []
    layers = []

    for name, param in model.named_parameters():
        if param.requires_grad and "bias" not in name:
            layers.append(name)
            ave_grads.append(param.grad.abs().mean().item())

    plt.figure(figsize=(10,5))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation=90)
    plt.ylabel("Average Gradient")
    plt.title("Gradient Flow")
    plt.tight_layout()
    return plt


In [9]:
def plot_weight_flow(model):
    weights = []
    for param in model.parameters():
        weights.append(param.data.norm().item())

    plt.figure(figsize=(6,4))
    plt.plot(weights)
    plt.title("Weight Update Flow")
    plt.ylabel("Weight Norm")
    return plt


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x,y in train_loader:
        x,y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(1)
        correct += (preds==y).sum().item()
        total += y.size(0)

    acc = correct/total
    avg_loss = total_loss/len(train_loader)

    wandb.log({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_accuracy": acc
    })

    # Gradient plot
    grad_fig = plot_grad_flow(model)
    wandb.log({"Gradient Flow": wandb.Image(grad_fig)})
    plt.close()

    # Weight plot
    weight_fig = plot_weight_flow(model)
    wandb.log({"Weight Flow": wandb.Image(weight_fig)})
    plt.close()

    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Acc={acc:.4f}")


Epoch 1: Loss=1.3116, Acc=0.5303
Epoch 2: Loss=0.9144, Acc=0.6780
Epoch 3: Loss=0.7420, Acc=0.7382
Epoch 4: Loss=0.5997, Acc=0.7896
Epoch 5: Loss=0.4738, Acc=0.8343
Epoch 6: Loss=0.3566, Acc=0.8745
Epoch 7: Loss=0.2587, Acc=0.9103
Epoch 8: Loss=0.1741, Acc=0.9412
Epoch 9: Loss=0.1229, Acc=0.9590
Epoch 10: Loss=0.0948, Acc=0.9676
Epoch 11: Loss=0.0808, Acc=0.9726
Epoch 12: Loss=0.0708, Acc=0.9766
Epoch 13: Loss=0.0561, Acc=0.9810
Epoch 14: Loss=0.0636, Acc=0.9783
Epoch 15: Loss=0.0637, Acc=0.9780
Epoch 16: Loss=0.0483, Acc=0.9836
Epoch 17: Loss=0.0507, Acc=0.9825
Epoch 18: Loss=0.0501, Acc=0.9829
Epoch 19: Loss=0.0446, Acc=0.9852
Epoch 20: Loss=0.0399, Acc=0.9864
Epoch 21: Loss=0.0452, Acc=0.9844
Epoch 22: Loss=0.0375, Acc=0.9874
Epoch 23: Loss=0.0432, Acc=0.9859
Epoch 24: Loss=0.0401, Acc=0.9869
Epoch 25: Loss=0.0364, Acc=0.9875
Epoch 26: Loss=0.0340, Acc=0.9889
Epoch 27: Loss=0.0369, Acc=0.9874
Epoch 28: Loss=0.0379, Acc=0.9878
Epoch 29: Loss=0.0353, Acc=0.9881
Epoch 30: Loss=0.0347, 

In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for x,y in test_loader:
        x,y = x.to(device), y.to(device)
        out = model(x)
        preds = out.argmax(1)
        correct += (preds==y).sum().item()
        total += y.size(0)

test_acc = correct/total
print("Test Accuracy:", test_acc)

wandb.log({"test_accuracy": test_acc})


Test Accuracy: 0.7231
